# SESSION 2.1: TENSORS & OPERATIONS — THEORY

---

## **WHAT IS A TENSOR?**

### **Simple Definition**
A tensor is a **multi-dimensional array** of numbers. Think of it as a generalization:

- **0D Tensor (Scalar):** Just a single number → `5`
- **1D Tensor (Vector):** A list of numbers → `[1, 2, 3, 4]`
- **2D Tensor (Matrix):** A table of numbers → `[[1, 2], [3, 4]]`
- **3D Tensor:** A cube of numbers → Think of a video frame (height × width × color channels)
- **4D Tensor:** A batch of video frames → (batch × height × width × channels)

### **Visual Example**
```
Scalar (0D):
    5

Vector (1D):
    [1, 2, 3, 4, 5]

Matrix (2D):
    [[1, 2, 3],
     [4, 5, 6]]

3D Tensor:
    [[[1, 2],    First "slice"
      [3, 4]],
     
     [[5, 6],    Second "slice"
      [7, 8]]]
```

---

## **TENSORS IN AUDIO PROCESSING**

### **Your Audio Data as Tensors**

When you load audio with librosa, you get a **1D numpy array**:
```python
audio = [0.01, 0.02, -0.03, 0.04, ...]  # 16,000 samples = 1 second
# Shape: (16000,)
```

But neural networks need **batched, multi-channel data**:
```python
# Single audio sample for training:
Shape: (1, 1, 16000)
        │  │    └── samples (time dimension)
        │  └── channels (mono = 1, stereo = 2)
        └── batch size (1 audio clip)

# Batch of 32 audio samples:
Shape: (32, 1, 16000)
        └── 32 different audio clips processed together
```

### **Why Batching?**
- **Efficiency:** Process multiple samples simultaneously using GPU parallelism
- **Stable gradients:** Average gradients across batch reduces noise
- **Faster training:** 32 samples at once is much faster than 32 × 1

---

## **TENSOR vs NUMPY ARRAY**

### **Similarities**
- Both are multi-dimensional arrays
- Both support indexing, slicing, reshaping
- Both support mathematical operations

### **Key Differences**

| Feature | NumPy Array | PyTorch Tensor |
|---------|-------------|----------------|
| **Device** | CPU only | CPU or GPU |
| **Gradients** | No automatic differentiation | Built-in autograd |
| **Deep Learning** | Not designed for it | Optimized for neural networks |
| **Speed** | Fast on CPU | Fast on both CPU/GPU |

### **Why PyTorch for Deep Learning?**

**1. Automatic Differentiation (Autograd)**
```python
# PyTorch tracks operations for automatic gradient computation
x = torch.tensor([2.0], requires_grad=True)
y = x ** 2  # y = 4
y.backward()  # Compute dy/dx automatically
print(x.grad)  # Output: tensor([4.0])  because dy/dx = 2x = 2(2) = 4
```

NumPy can't do this! You'd have to manually compute gradients.

**2. GPU Acceleration**
```python
# Move tensor to GPU (if available)
tensor_gpu = tensor.to('cuda')
# Now all operations are 10-100x faster!
```

---

## **TENSOR SHAPES & DIMENSIONS**

### **Understanding Shape**

```python
tensor = torch.tensor([[1, 2, 3],
                       [4, 5, 6]])

print(tensor.shape)  # torch.Size([2, 3])
                     #              │  └── 3 columns
                     #              └── 2 rows

print(tensor.dim())  # 2 (it's a 2D tensor)
print(tensor.size(0))  # 2 (size of dimension 0)
print(tensor.size(1))  # 3 (size of dimension 1)
```

### **Audio Example**
```python
# Single mono audio clip (1 second at 16kHz)
audio = torch.randn(16000)
print(audio.shape)  # torch.Size([16000])
print(audio.dim())  # 1 (1D tensor)

# Reshape for neural network input
audio_batched = audio.view(1, 1, 16000)
print(audio_batched.shape)  # torch.Size([1, 1, 16000])
print(audio_batched.dim())  # 3 (3D tensor)
```

---

## **COMMON TENSOR OPERATIONS**

### **1. Reshaping**

Changing shape without changing data:

```python
x = torch.tensor([1, 2, 3, 4, 5, 6])  # Shape: (6,)

# Reshape to 2x3 matrix
x_2d = x.view(2, 3)
# [[1, 2, 3],
#  [4, 5, 6]]

# Reshape to 3x2 matrix
x_3x2 = x.view(3, 2)
# [[1, 2],
#  [3, 4],
#  [5, 6]]

# Use -1 to auto-calculate one dimension
x_auto = x.view(2, -1)  # Same as view(2, 3)
```

**Important:** `view()` requires contiguous memory. Use `reshape()` if not sure.

### **2. Adding Dimensions (Unsqueeze)**

Adding a dimension of size 1:

```python
x = torch.tensor([1, 2, 3])  # Shape: (3,)

x_unsqueeze_0 = x.unsqueeze(0)  # Shape: (1, 3)
# [[1, 2, 3]]  ← Now it's a 2D tensor

x_unsqueeze_1 = x.unsqueeze(1)  # Shape: (3, 1)
# [[1],
#  [2],
#  [3]]  ← Also 2D, but different shape
```

**Audio Example:**
```python
audio = torch.randn(16000)  # Shape: (16000,)

# Add batch dimension
audio = audio.unsqueeze(0)  # Shape: (1, 16000)

# Add channel dimension
audio = audio.unsqueeze(0)  # Shape: (1, 1, 16000)
# Now ready for Conv1d!
```

### **3. Removing Dimensions (Squeeze)**

Removing dimensions of size 1:

```python
x = torch.tensor([[[1, 2, 3]]])  # Shape: (1, 1, 3)

x_squeezed = x.squeeze()  # Shape: (3,)
# [1, 2, 3]  ← All size-1 dimensions removed
```

### **4. Concatenation**

Joining tensors along a dimension:

```python
audio1 = torch.randn(1, 1, 8000)  # 0.5 seconds
audio2 = torch.randn(1, 1, 8000)  # 0.5 seconds

# Concatenate along time dimension (dim=2)
combined = torch.cat([audio1, audio2], dim=2)
print(combined.shape)  # torch.Size([1, 1, 16000])
# Now you have 1 second of audio!
```

---

## **GRADIENTS & AUTOGRAD**

This is what makes PyTorch powerful for deep learning!

### **The Concept**

In machine learning, we need to compute **how loss changes with respect to each parameter**:

```
If Loss = f(weight), then we need: ∂Loss/∂weight
```

This tells us which direction to adjust weights to reduce loss.

### **Manual vs Automatic**

**Manual (painful):**
```python
# You have to derive and code the gradient formula
def forward(x, w):
    return x * w

def backward(x, w):
    # Manually computed: d(x*w)/dw = x
    return x  

# For complex networks, this is VERY hard!
```

**Automatic (PyTorch):**
```python
x = torch.tensor([2.0])
w = torch.tensor([3.0], requires_grad=True)  # Track gradients

y = x * w  # Forward pass: y = 6.0
y.backward()  # Compute gradients automatically

print(w.grad)  # tensor([2.0])  ← PyTorch computed this!
```

### **How Autograd Works (Simplified)**

PyTorch builds a **computational graph** tracking all operations:

```
        x (2.0)    w (3.0, requires_grad=True)
           │         │
           └────*────┘  ← Multiplication operation
                │
              y (6.0)
```

When you call `y.backward()`:
1. Start at y
2. Walk backward through the graph
3. Apply chain rule at each operation
4. Accumulate gradients in `.grad` attributes

### **Important Gradient Concepts**

**1. requires_grad=True**
```python
# Only tensors with requires_grad=True track gradients
x = torch.tensor([1.0])  # requires_grad=False (default)
w = torch.tensor([2.0], requires_grad=True)  # Will track gradients

y = x * w
# y inherits requires_grad=True because w has it
```

**2. Gradient Accumulation**
```python
w = torch.tensor([1.0], requires_grad=True)

y1 = w * 2
y1.backward()
print(w.grad)  # tensor([2.0])

y2 = w * 3
y2.backward()
print(w.grad)  # tensor([5.0])  ← Accumulated! (2 + 3)
```

**3. Zero Gradients**
```python
# Clear gradients before next backward pass
w.grad.zero_()

y3 = w * 4
y3.backward()
print(w.grad)  # tensor([4.0])  ← Fresh gradient
```

This is crucial in training loops!

---

## **TENSOR DATA TYPES**

### **Common Types**

```python
# Integer types
torch.int32      # 32-bit integer
torch.int64      # 64-bit integer (torch.long)

# Float types
torch.float32    # 32-bit float (torch.float) - MOST COMMON
torch.float64    # 64-bit float (torch.double)

# Boolean
torch.bool       # True/False
```

### **Why float32?**
- ✅ Sufficient precision for neural networks
- ✅ Half the memory of float64
- ✅ Faster computation
- ✅ Standard in deep learning

### **Type Conversion**
```python
x = torch.tensor([1, 2, 3])  # Default: int64
x_float = x.float()          # Convert to float32
x_double = x.double()        # Convert to float64
```

---

## **CPU vs GPU (CUDA)**

### **Device Management**

```python
# Check if GPU is available
print(torch.cuda.is_available())  # True if you have NVIDIA GPU

# Create tensor on CPU (default)
x_cpu = torch.tensor([1, 2, 3])

# Move to GPU
if torch.cuda.is_available():
    x_gpu = x_cpu.to('cuda')
    # or
    x_gpu = x_cpu.cuda()
    
# Move back to CPU
x_back = x_gpu.to('cpu')
# or
x_back = x_gpu.cpu()
```

### **Important Rules**
- ⚠️ **All tensors in an operation must be on the same device**
- ⚠️ **NumPy only works with CPU tensors**

```python
# This will error:
x_cpu = torch.tensor([1, 2])
x_gpu = torch.tensor([3, 4]).cuda()
result = x_cpu + x_gpu  # ❌ Error! Different devices

# Fix: Move both to same device
result = x_cpu.cuda() + x_gpu  # ✅ Works
```

---

## **KEY TAKEAWAYS**

1. **Tensors are multi-dimensional arrays** optimized for deep learning
2. **Shape matters:** Neural networks expect specific tensor shapes
3. **Batching:** Process multiple samples together for efficiency
4. **Autograd:** PyTorch automatically computes gradients for backpropagation
5. **float32:** Standard data type for neural networks
6. **Device consistency:** Keep tensors on same device (CPU or GPU)

---

## **AUDIO-SPECIFIC SUMMARY**

For your voice isolation project:

```python
# Input: NumPy array from librosa
audio_numpy = librosa.load(...)  # Shape: (16000,)

# Convert to tensor
audio_tensor = torch.from_numpy(audio_numpy)  # Shape: (16000,)

# Reshape for Conv1d input: (batch, channels, samples)
audio_tensor = audio_tensor.unsqueeze(0).unsqueeze(0)  # Shape: (1, 1, 16000)

# Now ready for your model!
```

**EXCELLENT QUESTIONS!** 🎯

Let me answer both before giving you the tasks.

---

## **QUESTION 1: RESHAPING RULE**

### **YES! You're exactly right!**

**The Rule:**
```
Total number of elements must remain the same
```

**Math:**
```python
original: (6,)     → 6 elements total
reshape:  (2, 3)   → 2 × 3 = 6 elements ✅
reshape:  (3, 2)   → 3 × 2 = 6 elements ✅
reshape:  (1, 6)   → 1 × 6 = 6 elements ✅
reshape:  (6, 1)   → 6 × 1 = 6 elements ✅

# This would ERROR:
reshape:  (2, 2)   → 2 × 2 = 4 elements ❌ (6 ≠ 4)
```

**Audio Example:**
```python
audio = torch.randn(16000)  # 16,000 elements

# Valid reshapes:
audio.view(1, 16000)        # 1 × 16,000 = 16,000 ✅
audio.view(1, 1, 16000)     # 1 × 1 × 16,000 = 16,000 ✅
audio.view(2, 8000)         # 2 × 8,000 = 16,000 ✅
audio.view(100, 160)        # 100 × 160 = 16,000 ✅

# Invalid:
audio.view(1, 10000)        # 1 × 10,000 = 10,000 ❌ ERROR!
```

**The -1 Trick:**
```python
# Use -1 to auto-calculate one dimension
audio.view(1, 1, -1)   # PyTorch calculates: -1 = 16000/1/1 = 16000
audio.view(2, -1)      # PyTorch calculates: -1 = 16000/2 = 8000
audio.view(-1, 160)    # PyTorch calculates: -1 = 16000/160 = 100
```

---

## **QUESTION 2: GPU HANDLING**

### **A. Different GPU Types**

**NVIDIA GPUs (CUDA):**
```python
device = torch.device('cuda')  # NVIDIA GPU
```

**Apple Silicon (M1/M2/M3) - MPS:**
```python
device = torch.device('mps')   # Apple Metal Performance Shaders
```

**AMD GPUs (ROCm):**
```python
device = torch.device('cuda')  # AMD with ROCm (same API as NVIDIA)
```

**CPU (Fallback):**
```python
device = torch.device('cpu')
```

---

### **B. Automatic Device Selection**

**Smart Device Selection:**
```python
# Automatically choose best available device
if torch.cuda.is_available():
    device = torch.device('cuda')      # NVIDIA GPU
elif torch.backends.mps.is_available():
    device = torch.device('mps')       # Apple Silicon
else:
    device = torch.device('cpu')       # Fallback

print(f"Using device: {device}")
```

---

### **C. Default Tensor Creation on GPU**

**Method 1: Set Default Device (Explicit)**
```python
# NOT a built-in PyTorch feature, but you can create a helper:

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

def tensor(*args, **kwargs):
    """Create tensor on default device"""
    return torch.tensor(*args, **kwargs).to(device)

# Now use your custom function:
x = tensor([1, 2, 3])  # Created on GPU automatically!
print(x.device)  # cuda:0
```

**Method 2: Wrapper Class (Cleaner)**
```python
class TensorFactory:
    def __init__(self):
        if torch.cuda.is_available():
            self.device = torch.device('cuda')
        elif torch.backends.mps.is_available():
            self.device = torch.device('mps')
        else:
            self.device = torch.device('cpu')
    
    def tensor(self, *args, **kwargs):
        return torch.tensor(*args, **kwargs, device=self.device)
    
    def randn(self, *args, **kwargs):
        return torch.randn(*args, **kwargs, device=self.device)
    
    def zeros(self, *args, **kwargs):
        return torch.zeros(*args, **kwargs, device=self.device)

# Usage:
tf = TensorFactory()
x = tf.tensor([1, 2, 3])      # On GPU automatically
y = tf.randn(100, 100)        # On GPU automatically
z = tf.zeros(10, 10)          # On GPU automatically
```

**Method 3: PyTorch's Built-in `set_default_tensor_type()` (Not Recommended)**
```python
# This changes the DEFAULT type globally
torch.set_default_tensor_type('torch.cuda.FloatTensor')

# Now tensors are created on GPU by default
x = torch.tensor([1, 2, 3])  # On GPU!

# ⚠️ Problem: This can cause unexpected behavior in libraries
# ⚠️ Not recommended for production code
```

---

### **D. Best Practice (What I Recommend)**

**Create a config at the top of your notebook/script:**

```python
# At the start of your notebook
import torch

# Device configuration
if torch.cuda.is_available():
    device = torch.device('cuda')
    print(f"✅ Using NVIDIA GPU: {torch.cuda.get_device_name(0)}")
elif torch.backends.mps.is_available():
    device = torch.device('mps')
    print("✅ Using Apple Silicon GPU (MPS)")
else:
    device = torch.device('cpu')
    print("⚠️  Using CPU (no GPU available)")

# Helper function for easy tensor creation
def to_device(tensor):
    return tensor.to(device)

# Usage throughout your code:
x = torch.tensor([1, 2, 3]).to(device)
# or
x = to_device(torch.tensor([1, 2, 3]))

# For model:
model = MyModel().to(device)
```

---

### **E. For Your MacBook Specifically**

If you have Apple Silicon (M1/M2/M3):

```python
import torch

# Check MPS availability
if torch.backends.mps.is_available():
    if not torch.backends.mps.is_built():
        print("❌ MPS not available because PyTorch not built with MPS")
    else:
        device = torch.device("mps")
        print("✅ Using Apple Silicon GPU (MPS)")
else:
    device = torch.device("cpu")
    print("⚠️  MPS not available, using CPU")

# Test it
x = torch.randn(5, 5).to(device)
print(x.device)  # mps:0 or cpu
```

**Note for Apple Silicon:**
- MPS support was added in PyTorch 1.12+
- Not all operations are supported on MPS yet
- Some operations automatically fall back to CPU

---

### **F. Summary: Device Management Strategy**

```python
# ✅ RECOMMENDED PATTERN:

# 1. Set device once at the top
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

# 2. Move data to device when loading
for batch in dataloader:
    audio, labels = batch
    audio = audio.to(device)
    labels = labels.to(device)
    
    # Now everything is on the same device

# 3. Move model to device
model = MyModel().to(device)

# 4. All operations happen on the same device automatically
output = model(audio)  # Both model and audio on GPU → fast!
```

# ============================================
# SESSION 2.1: TENSORS & OPERATIONS
# Part A: Tensor Basics
# ============================================

In [30]:
import torch
import numpy as np
from pathlib import Path

# Device configuration (for your MacBook)
if torch.backends.mps.is_available():
    device = torch.device('mps')
    print("✅ Using Apple Silicon GPU")
elif torch.cuda.is_available():
    device = torch.device('cuda')
    print("✅ Using NVIDIA GPU")
else:
    device = torch.device('cpu')
    print("⚠️ Using CPU")

print(f"PyTorch version: {torch.__version__}")
print(f"Device: {device}")

✅ Using Apple Silicon GPU
PyTorch version: 2.10.0
Device: mps


## TODO 1: Create Your First Tensor

In [31]:
# TODO 1.1: Create a 1D tensor with values [1, 2, 3, 4, 5]
# Hint: Use torch.tensor()

tensor_1d = torch.tensor([1, 2, 3, 4, 5])

# Test your answer
print("TODO 1.1:")
print(f"Tensor: {tensor_1d}")
print(f"Shape: {tensor_1d.shape}")
print(f"Dimensions: {tensor_1d.dim()}")
print(f"Data type: {tensor_1d.dtype}")

# Expected output:
# Tensor: tensor([1, 2, 3, 4, 5])
# Shape: torch.Size([5])
# Dimensions: 1
# Data type: torch.int64

TODO 1.1:
Tensor: tensor([1, 2, 3, 4, 5])
Shape: torch.Size([5])
Dimensions: 1
Data type: torch.int64


In [32]:
# TODO 1.2: Create a 2D tensor (matrix) with shape (3, 4) containing random values
# Hint: Use torch.randn() which creates random values from normal distribution

tensor_2d = torch.randn([3,4])

# Test your answer
print("\nTODO 1.2:")
print(f"Tensor:\n{tensor_2d}")
print(f"Shape: {tensor_2d.shape}")
print(f"Dimensions: {tensor_2d.dim()}")

# Expected output:
# Shape: torch.Size([3, 4])
# Dimensions: 2


TODO 1.2:
Tensor:
tensor([[-0.5779, -1.1044,  0.6975,  1.8363],
        [-0.6273,  1.0269,  1.5625,  0.0910],
        [ 2.1042, -0.5486, -0.0226,  0.0490]])
Shape: torch.Size([3, 4])
Dimensions: 2


In [33]:
# TODO 1.3: Create a tensor of zeros with shape (2, 3, 4)
# Hint: Use torch.zeros()

tensor_zeros = torch.zeros([2,3,4])

# Test your answer
print("\nTODO 1.3:")
print(f"Shape: {tensor_zeros.shape}")
print(f"All zeros? {torch.all(tensor_zeros == 0)}")

# Expected: Shape: torch.Size([2, 3, 4])
# Expected: All zeros? True


TODO 1.3:
Shape: torch.Size([2, 3, 4])
All zeros? True


## TODO 2: NumPy ↔ PyTorch Conversion

In [34]:
# TODO 2.1: Convert a NumPy array to PyTorch tensor
numpy_array = np.array([1.0, 2.0, 3.0, 4.0, 5.0])

# TODO: Convert to tensor
# Hint: Use torch.from_numpy()
tensor_from_numpy = torch.from_numpy(numpy_array)

# Test your answer
print("\nTODO 2.1:")
# if numpy is not converted to tensor then rase a error
print(f"Original NumPy type: {type(numpy_array)}")
print(f"Converted type: {type(tensor_from_numpy)}")
print(f"Values match: {np.array_equal(numpy_array, tensor_from_numpy.numpy())}")

assert isinstance(tensor_from_numpy, torch.Tensor), "Should be a PyTorch tensor!"


TODO 2.1:
Original NumPy type: <class 'numpy.ndarray'>
Converted type: <class 'torch.Tensor'>
Values match: True


In [35]:
# TODO 2.2: Convert a PyTorch tensor back to NumPy
torch_tensor = torch.tensor([10, 20, 30, 40, 50])

# TODO: Convert to numpy
# Hint: Use .numpy() method
numpy_from_torch = torch_tensor.numpy()

# Test your answer
print("\nTODO 2.2:")
print(f"Tensor type: {type(torch_tensor)}")
print(f"NumPy type: {type(numpy_from_torch)}")
print(f"Values: {numpy_from_torch}")

assert isinstance(numpy_from_torch, np.ndarray), "Should be a NumPy array!"


TODO 2.2:
Tensor type: <class 'torch.Tensor'>
NumPy type: <class 'numpy.ndarray'>
Values: [10 20 30 40 50]


## TODO 3: Load Real Audio and Convert to Tensor

In [36]:
import sys
sys.path.insert(0, str(Path.cwd().parent.parent))  # Add project root to path

from src.preprocessing.audio_utils import load_audio

# TODO 3.1: Load one of your audio files
# Find any .flac file in your data/raw directory
data_dir = Path.cwd().parent.parent / "data" / "raw"
audio_files = list(data_dir.rglob("*.flac"))

if len(audio_files) == 0:
    print("❌ No audio files found! Check your data/raw directory")
else:
    # Take first file
    audio_file = audio_files[0]
    print(f"Loading: {audio_file.name}")
    
    # TODO: Load audio using your audio_utils function
    # Hint: You already wrote this function!
    audio_numpy, sr = load_audio(file_path=audio_file)
    
    print(f"\nTODO 3.1:")
    print(f"Audio shape (NumPy): {audio_numpy.shape}")
    print(f"Sample rate: {sr} Hz")
    print(f"Duration: {len(audio_numpy) / sr:.2f} seconds")
    print(f"Data type: {audio_numpy.dtype}")

Loading: 2412-153954-0019.flac

TODO 3.1:
Audio shape (NumPy): (119760,)
Sample rate: 16000 Hz
Duration: 7.49 seconds
Data type: float32


In [37]:
# TODO 3.2: Convert audio to PyTorch tensor
# TODO: Convert the audio_numpy array to a PyTorch tensor
# Hint: Use torch.from_numpy()

audio_tensor = torch.from_numpy(audio_numpy)

# Test your answer
print("\nTODO 3.2:")
print(f"Tensor shape: {audio_tensor.shape}")
print(f"Tensor dtype: {audio_tensor.dtype}")
print(f"Min value: {audio_tensor.min():.4f}")
print(f"Max value: {audio_tensor.max():.4f}")

assert audio_tensor.dtype == torch.float32, "Should be float32!"
assert audio_tensor.dim() == 1, "Should be 1D!"


TODO 3.2:
Tensor shape: torch.Size([119760])
Tensor dtype: torch.float32
Min value: -0.6028
Max value: 0.3453


## TODO 4: Reshape Audio for Neural Networks

In [38]:
# Neural networks expect: (batch_size, channels, samples)
# Your audio_tensor is currently: (samples,)

# TODO 4.1: Add batch dimension using unsqueeze
# The audio_tensor is shape (16000,) or similar
# Add dimension at position 0 to make it (1, 16000)
# Hint: Use .unsqueeze(0)

audio_with_batch = audio_tensor.unsqueeze(0)

print("\nTODO 4.1:")
print(f"Original shape: {audio_tensor.shape}")
print(f"With batch: {audio_with_batch.shape}")

assert audio_with_batch.shape[0] == 1, "First dimension should be 1 (batch size)"


TODO 4.1:
Original shape: torch.Size([119760])
With batch: torch.Size([1, 119760])


In [39]:
# TODO 4.2: Add channel dimension
# Now you have (1, samples)
# Add another dimension to make it (1, 1, samples)
# Hint: Use .unsqueeze(1) on audio_with_batch

audio_final = audio_with_batch.unsqueeze(0)

print("\nTODO 4.2:")
print(f"Previous shape: {audio_with_batch.shape}")
print(f"Final shape: {audio_final.shape}")
print(f"This is ready for Conv1d!")

assert audio_final.dim() == 3, "Should be 3D tensor"
assert audio_final.shape[0] == 1, "Batch size should be 1"
assert audio_final.shape[1] == 1, "Channels should be 1 (mono)"



TODO 4.2:
Previous shape: torch.Size([1, 119760])
Final shape: torch.Size([1, 1, 119760])
This is ready for Conv1d!


In [40]:
# TODO 4.3: Alternative method using view/reshape
# Do the same reshaping in ONE step using .view() or .reshape()
# Start from audio_tensor (1D) and make it (1, 1, samples)
# Hint: Use .view(1, 1, -1) where -1 auto-calculates the last dimension

audio_onestep = audio_tensor.view(1,1,-1)

print("\nTODO 4.3:")
print(f"Original shape: {audio_tensor.shape}")
print(f"One-step reshape: {audio_onestep.shape}")
print(f"Same as previous method? {torch.equal(audio_final, audio_onestep)}")

assert audio_onestep.shape == audio_final.shape, "Should match previous result"


TODO 4.3:
Original shape: torch.Size([119760])
One-step reshape: torch.Size([1, 1, 119760])
Same as previous method? True


# ============================================
# Part B: Tensor Operations
# ============================================

## TODO 5: Concatenation (Combining Audio)


In [41]:
# Imagine you want to combine multiple audio clips into one batch

# TODO 5.1: Load 3 different audio files and convert to tensors
# Each should have shape (1, 1, samples)

data_dir = Path.cwd().parent.parent / "data" / "raw"
audio_files = list(data_dir.rglob("*.flac"))[:3]  # Take first 3 files

# TODO: Load each file, convert to tensor, and reshape to (1, 1, samples)
# Store them in a list
# Hint: Use a loop or list comprehension

audio_tensors = []

# YOUR CODE HERE
for audio in audio_files:
    audio_numpy, sr = load_audio(file_path=audio)
    audio_tensor = torch.from_numpy(audio_numpy)
    audio_final = audio_tensor.view(1,1,-1)  # Reshape to (1, 1, samples)
    audio_tensors.append(audio_final)

print("\nTODO 5.1:")
for i, tensor in enumerate(audio_tensors):
    print(f"Audio {i+1} shape: {tensor.shape}")

assert len(audio_tensors) == 3, "Should have 3 audio tensors"


TODO 5.1:
Audio 1 shape: torch.Size([1, 1, 119760])
Audio 2 shape: torch.Size([1, 1, 42400])
Audio 3 shape: torch.Size([1, 1, 182240])


In [42]:
# TODO 5.2: Concatenate along the batch dimension
# Problem: Your audio files have different lengths!
# Solution: Pad them to the same length first

import torch.nn.functional as F

# Step 1: Find the maximum length
max_length = max(tensor.shape[2] for tensor in audio_tensors)

# Step 2: Pad each tensor to max_length
# TODO: Complete this loop
padded_tensors = []
for tensor in audio_tensors:
    current_length = tensor.shape[2]
    padding_needed = max_length - current_length
    
    # TODO: Use F.pad to add zeros on the right
    # Hint: F.pad(tensor, (left_pad, right_pad))
    padded = F.pad(tensor, (0, padding_needed))
    
    padded_tensors.append(padded)

# Step 3: Now concatenate
batch_of_audio = torch.cat(tensors=padded_tensors, dim=0)

print("\nTODO 5.2:")
print(f"Max length: {max_length} samples")
print(f"Padded shapes:")
for i, tensor in enumerate(padded_tensors):
    print(f"  Audio {i+1}: {tensor.shape}")
print(f"Concatenated shape: {batch_of_audio.shape}")

assert batch_of_audio.shape[0] == 3, "Batch size should be 3"
assert batch_of_audio.dim() == 3, "Should still be 3D"
assert batch_of_audio.shape[2] == max_length, "All should be padded to max length"


TODO 5.2:
Max length: 182240 samples
Padded shapes:
  Audio 1: torch.Size([1, 1, 182240])
  Audio 2: torch.Size([1, 1, 182240])
  Audio 3: torch.Size([1, 1, 182240])
Concatenated shape: torch.Size([3, 1, 182240])


In [43]:
# TODO 5.3: What if audio files have different lengths?
# Let's simulate this by trimming one audio file

audio_short = audio_tensors[0][:, :, :8000]  # Only first 0.5 seconds
audio_long = audio_tensors[1]  # Full length

print("\nTODO 5.3:")
print(f"Short audio shape: {audio_short.shape}")
print(f"Long audio shape: {audio_long.shape}")

# TODO: Try to concatenate them
# What happens? Will this work?

try:
    combined = torch.cat(tensors=(audio_short, audio_long), dim=0)
    print(f"✅ Concatenation worked: {combined.shape}")
except Exception as e:
    print(f"❌ Error: {e}")
    print("Why did this fail? Think about the requirement for concatenation.")


TODO 5.3:
Short audio shape: torch.Size([1, 1, 8000])
Long audio shape: torch.Size([1, 1, 42400])
❌ Error: Sizes of tensors must match except in dimension 0. Expected size 8000 but got size 42400 for tensor number 1 in the list.
Why did this fail? Think about the requirement for concatenation.


## TODO 6: Slicing and Indexing


In [44]:
# TODO 6.1: Extract a specific time segment from audio
# Let's say you want seconds 2-3 from a 5-second audio clip

# Create a sample 5-second audio (5 * 16000 = 80000 samples)
audio_5sec = torch.randn(1, 1, 80000)

# TODO: Extract samples from 2.0 to 3.0 seconds
# Sample rate = 16000 Hz
# Start index = 2 * 16000 = 32000
# End index = 3 * 16000 = 48000
# Hint: Use slicing audio_5sec[:, :, start:end]

sample_rate = 16000
start_idx = 2 * sample_rate
end_idx = 3 * sample_rate

segment_2_to_3 = audio_5sec[:, :, start_idx:end_idx]

print("\nTODO 6.1:")
print(f"Original audio: {audio_5sec.shape}")
print(f"Segment shape: {segment_2_to_3.shape}")
print(f"Segment duration: {segment_2_to_3.shape[2] / 16000} seconds")

assert segment_2_to_3.shape[2] == 16000, "Should be 1 second of audio"


TODO 6.1:
Original audio: torch.Size([1, 1, 80000])
Segment shape: torch.Size([1, 1, 16000])
Segment duration: 1.0 seconds


In [45]:
# TODO 6.2: Extract every other sample (downsample by 2)
# This effectively changes sample rate from 16kHz to 8kHz
# Hint: Use slicing with step: audio[::2] means every 2nd element

downsampled = audio_5sec[:, :, ::2]

print("\nTODO 6.2:")
print(f"Original shape: {audio_5sec.shape}")
print(f"Downsampled shape: {downsampled.shape}")
print(f"Original sample rate: 16000 Hz")
print(f"New effective sample rate: {16000 / 2} Hz")

assert downsampled.shape[2] == 40000, "Should be half the samples"


TODO 6.2:
Original shape: torch.Size([1, 1, 80000])
Downsampled shape: torch.Size([1, 1, 40000])
Original sample rate: 16000 Hz
New effective sample rate: 8000.0 Hz


In [46]:
# TODO 6.3: Get the first audio clip from a batch
# If you have batch_of_audio with shape (3, 1, samples)
# Extract just the first audio clip

first_audio = batch_of_audio[1, :, :]

print("\nTODO 6.3:")
print(f"Batch shape: {batch_of_audio.shape}")
print(f"First audio shape: {first_audio.shape}")

assert first_audio.shape[0] == 1, "Should have batch size 1"


TODO 6.3:
Batch shape: torch.Size([3, 1, 182240])
First audio shape: torch.Size([1, 182240])


## TODO 7: Element-wise Operations

In [47]:
# TODO 7.1: Normalize audio to [-1, 1] range
# Sometimes audio values exceed this range after mixing
# Formula: normalized = audio / max(abs(audio))

unnormalized_audio = torch.randn(1, 1, 16000) * 5  # Values between -5 and 5

# TODO: Normalize it
# Step 1: Find maximum absolute value
# Step 2: Divide audio by this maximum
# Hint: Use torch.max() and torch.abs()

max_val = torch.max(torch.abs(unnormalized_audio))
normalized_audio = unnormalized_audio/max_val

print("\nTODO 7.1:")
print(f"Before normalization:")
print(f"  Min: {unnormalized_audio.min():.2f}, Max: {unnormalized_audio.max():.2f}")
print(f"After normalization:")
print(f"  Min: {normalized_audio.min():.2f}, Max: {normalized_audio.max():.2f}")

assert normalized_audio.abs().max() <= 1.0, "Should be in [-1, 1] range"


TODO 7.1:
Before normalization:
  Min: -24.58, Max: 19.33
After normalization:
  Min: -1.00, Max: 0.79


In [48]:
# TODO 7.2: Add two audio signals together (mixing)
# This is how you create mixed audio from clean sources!

speaker1 = torch.randn(1, 1, 16000) * 0.5
speaker2 = torch.randn(1, 1, 16000) * 0.5

# TODO: Mix them by adding
mixture = speaker1 + speaker2

normalized_mixture = mixture/torch.max(torch.abs(mixture))

print("\nTODO 7.2:")
print(f"Speaker 1 shape: {speaker1.shape}")
print(f"Speaker 2 shape: {speaker2.shape}")
print(f"Mixture shape: {mixture.shape}")
print(f"Mixture max amplitude: {mixture.abs().max():.2f}")

# The mixture should have same shape as individual speakers
assert mixture.shape == speaker1.shape


TODO 7.2:
Speaker 1 shape: torch.Size([1, 1, 16000])
Speaker 2 shape: torch.Size([1, 1, 16000])
Mixture shape: torch.Size([1, 1, 16000])
Mixture max amplitude: 2.82


In [49]:
# TODO 7.3: Apply a mask to audio (this is what separation models do!)
# A mask is values between 0 and 1
# Multiplying audio by a mask isolates certain parts

audio = torch.randn(1, 1, 16000)
mask = torch.rand(1, 1, 16000)  # Random values between 0 and 1

# TODO: Apply mask by element-wise multiplication
masked_audio = audio * mask

print("\nTODO 7.3:")
print(f"Audio shape: {audio.shape}")
print(f"Mask shape: {mask.shape}")
print(f"Masked audio shape: {masked_audio.shape}")
print(f"Mask reduced energy: {masked_audio.abs().mean():.4f} < {audio.abs().mean():.4f}")

assert masked_audio.shape == audio.shape


TODO 7.3:
Audio shape: torch.Size([1, 1, 16000])
Mask shape: torch.Size([1, 1, 16000])
Masked audio shape: torch.Size([1, 1, 16000])
Mask reduced energy: 0.3991 < 0.8003


## TODO 8: Gradients (Introduction to Autograd)

In [50]:
# TODO 8.1: Enable gradient tracking
# Create a simple "weight" parameter that requires gradients

# TODO: Create a tensor with requires_grad=True
weight = torch.randn((5,3), requires_grad=True)

print("\nTODO 8.1:")
print(f"Weight value: {weight}")
print(f"Requires grad? {weight.requires_grad}")
print(f"Gradient: {weight.grad}")  # Should be None initially

assert weight.requires_grad == True, "Should track gradients"


TODO 8.1:
Weight value: tensor([[-0.2344,  0.6416, -0.1887],
        [-0.5699, -0.6561,  0.7790],
        [ 0.1767, -0.3775,  1.1506],
        [ 0.2938,  0.1193,  0.2032],
        [ 0.3145,  0.1726,  0.8148]], requires_grad=True)
Requires grad? True
Gradient: None


In [51]:
# TODO 8.2: Compute a simple gradient
# Let's compute: y = weight^2
# Then find dy/dweight

weight = torch.tensor([3.0], requires_grad=True)

# TODO: Compute y = weight^2
y = weight**2

# TODO: Compute gradients
# Hint: Call .backward() on y
y.backward()

print("\nTODO 8.2:")
print(f"Weight: {weight.item()}")
print(f"y = weight^2 = {y.item()}")
print(f"Gradient (dy/dweight): {weight.grad.item()}")
print(f"Expected gradient (2*weight = 2*3): 6.0")

# Mathematical check: d(x^2)/dx = 2x, so d(3^2)/d3 = 2*3 = 6
assert abs(weight.grad.item() - 6.0) < 0.001, "Gradient should be 6.0"


TODO 8.2:
Weight: 3.0
y = weight^2 = 9.0
Gradient (dy/dweight): 6.0
Expected gradient (2*weight = 2*3): 6.0


In [52]:
# TODO 8.3: Gradient accumulation and zeroing
# This is crucial for training loops!

weight = torch.tensor([2.0], requires_grad=True)

# First computation
y1 = weight * 3
y1.backward()
print("\nTODO 8.3:")
print(f"After first backward, gradient: {weight.grad.item()}")

# Second computation (without zeroing)
y2 = weight * 5
y2.backward()
print(f"After second backward, gradient: {weight.grad.item()}")
print(f"Notice: gradients accumulated! (3 + 5 = 8)")

# TODO: Zero the gradient
# Hint: weight.grad.zero_()
weight.grad.zero_()

print(f"After zeroing: {weight.grad}")

# Third computation (after zeroing)
y3 = weight * 7
y3.backward()
print(f"After third backward, gradient: {weight.grad.item()}")
print(f"Now it's just 7 (not accumulated)")


TODO 8.3:
After first backward, gradient: 3.0
After second backward, gradient: 8.0
Notice: gradients accumulated! (3 + 5 = 8)
After zeroing: tensor([0.])
After third backward, gradient: 7.0
Now it's just 7 (not accumulated)


## TODO 9: Moving Tensors to GPU

In [57]:
# TODO 9.1: Move a tensor to your device (MPS/CUDA/CPU)

cpu_tensor = torch.randn(1000, 1000)

# TODO: Move to the device you configured at the start
# Hint: Use .to(device)
device_tensor = cpu_tensor.to(device)

print("\nTODO 9.1:")
print(f"Original device: {cpu_tensor.device}")
print(f"New device: {device_tensor.device}")

assert device_tensor.device.type == torch.device(device).type, f"Should be on {device}"


TODO 9.1:
Original device: cpu
New device: mps:0


In [58]:
# TODO 9.2: Understand the device requirement
# Try to add tensors on different devices (this will error!)

cpu_tensor = torch.tensor([1, 2, 3])
device_tensor = torch.tensor([4, 5, 6]).to(device)

print("\nTODO 9.2:")
print(f"CPU tensor device: {cpu_tensor.device}")
print(f"Device tensor device: {device_tensor.device}")

# TODO: Try to add them (this will cause an error)
try:
    result = cpu_tensor + device_tensor
    print(f"✅ Addition worked: {result}")
except Exception as e:
    print(f"❌ Error: {e}")
    print("\nLesson: All tensors in an operation must be on the same device!")
    
    # TODO: Fix it by moving cpu_tensor to device
    result = cpu_tensor.to(device) + device_tensor
    print(f"✅ Fixed! Result: {result}")


TODO 9.2:
CPU tensor device: cpu
Device tensor device: mps:0
❌ Error: Expected all tensors to be on the same device, but found at least two devices, mps:0 and cpu!

Lesson: All tensors in an operation must be on the same device!
✅ Fixed! Result: tensor([5, 7, 9], device='mps:0')
